In [1]:
# Install required packages
!pip install fastapi uvicorn httpx python-dotenv

In [7]:
import os
from typing import Any, Dict

import httpx
from dotenv import load_dotenv
from fastapi import FastAPI, HTTPException, Query

load_dotenv()   # Load environment variables if any

app = FastAPI(
    title="Weather API",
    description="Fetch weather data from OpenWeather and Open-Meteo",
    version="1.0.0",
)

In [8]:
async def get_openweather_weather(lat: float, lon: float) -> Dict[str, Any]:
    """
    Fetch weather data from OpenWeather API.
    Returns only essential fields as JSON-compatible dict.
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")
    if not api_key:
        raise HTTPException(
            status_code=500,
            detail="OPENWEATHER_API_KEY is not set in environment."
        )

    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": api_key,
        "units": "metric",
        "lang": "en"
    }

    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.get(url, params=params)
    except httpx.RequestError:
        raise HTTPException(
            status_code=502,
            detail="Could not connect to OpenWeather service."
        )

    if response.status_code != 200:
        try:
            error_message = response.json().get("message", "Unknown error")
        except Exception:
            error_message = "Error fetching data from OpenWeather"
        raise HTTPException(
            status_code=502,
            detail=f"OpenWeather error: {error_message}"
        )

    data = response.json()
    return {
        "provider": "openweather",
        "latitude": lat,
        "longitude": lon,
        "location": data.get("name"),
        "temperature": data["main"]["temp"],
        "feels_like": data["main"]["feels_like"],
        "humidity": data["main"]["humidity"],
        "pressure": data["main"]["pressure"],
        "wind_speed": data.get("wind", {}).get("speed"),
        "weather": data["weather"][0]["description"]
    }

In [9]:
async def get_openmeteo_weather(lat: float, lon: float) -> Dict[str, Any]:
    """
    Fetch weather data from Open-Meteo API (no API key required).
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": (
            "temperature_2m,relative_humidity_2m,"
            "apparent_temperature,wind_speed_10m,weather_code"
        ),
        "timezone": "auto"
    }

    try:
        async with httpx.AsyncClient(timeout=10.0) as client:
            response = await client.get(url, params=params)
    except httpx.RequestError:
        raise HTTPException(
            status_code=502,
            detail="Could not connect to Open-Meteo service."
        )

    if response.status_code != 200:
        raise HTTPException(
            status_code=502,
            detail="Error fetching data from Open-Meteo"
        )

    data = response.json()
    current = data.get("current", {})
    units = data.get("current_units", {})

    return {
        "provider": "openmeteo",
        "latitude": lat,
        "longitude": lon,
        "timezone": data.get("timezone"),
        "temperature": current.get("temperature_2m"),
        "temperature_unit": units.get("temperature_2m"),
        "feels_like": current.get("apparent_temperature"),
        "humidity": current.get("relative_humidity_2m"),
        "wind_speed": current.get("wind_speed_10m"),
        "wind_speed_unit": units.get("wind_speed_10m"),
        "weather_code": current.get("weather_code"),
        "observation_time": current.get("time")
    }

In [10]:
@app.get("/weather")
async def get_weather(
    lat: float = Query(
        ...,
        description="Latitude, must be between -90 and 90",
        ge=-90,
        le=90
    ),
    lon: float = Query(
        ...,
        description="Longitude, must be between -180 and 180",
        ge=-180,
        le=180
    ),
    provider: str = Query(
        ...,
        description="Provider name: 'openweather' or 'openmeteo'"
    )
):
    """
    Get current weather for given coordinates.
    Validates input and calls the appropriate provider function.
    """
    provider = provider.lower().strip()
    if provider not in {"openweather", "openmeteo"}:
        raise HTTPException(
            status_code=400,
            detail="Invalid provider. Allowed: 'openweather', 'openmeteo'"
        )

    if provider == "openweather":
        return await get_openweather_weather(lat, lon)
    else:
        return await get_openmeteo_weather(lat, lon)

In [12]:
import asyncio
import threading
import uvicorn

# Create a new event loop for the thread
loop = asyncio.new_event_loop()

# Configure Uvicorn server
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)

# Function to run the server with the given loop
def run_server():
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

# Start the server in a background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server is running at http://127.0.0.1:8000")

Server is running at http://127.0.0.1:8000


In [13]:
import requests

def test_weather(lat, lon, provider):
    url = "http://127.0.0.1:8000/weather"
    resp = requests.get(url, params={"lat": lat, "lon": lon, "provider": provider})
    print(f"Status: {resp.status_code}")
    if resp.status_code == 200:
        print(resp.json())
    else:
        print(resp.json()["detail"])

# Test Open-Meteo (Tehran)
test_weather(35.6892, 51.3890, "openmeteo")

# Test invalid provider
test_weather(35.6892, 51.3890, "invalid")

# Test invalid lat (out of range) -> 422
test_weather(100, 51.3890, "openmeteo")

Status: 200
{'provider': 'openmeteo', 'latitude': 35.6892, 'longitude': 51.389, 'timezone': 'Asia/Tehran', 'temperature': 36.1, 'temperature_unit': '°C', 'feels_like': 35.0, 'humidity': 15, 'wind_speed': 9.4, 'wind_speed_unit': 'km/h', 'weather_code': 0, 'observation_time': '2026-08-19T10:00'}
Status: 400
Invalid provider. Allowed: 'openweather', 'openmeteo'
Status: 422
[{'type': 'less_than_equal', 'loc': ['query', 'lat'], 'msg': 'Input should be less than or equal to 90', 'input': '100', 'ctx': {'le': 90.0}}]
